In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader
import torchmetrics
import optuna

torch.manual_seed(42)
np.random.seed(42)

data = pd.read_csv("../../../data/raw/kathmandu_full_raw_2023_2024.csv")
data["time"] = pd.to_datetime(data["time"])
data = data.sort_values("time").reset_index(drop=True)

def cyclical(df, col, period):
    df[f"{col}_sin"] = np.sin(2 * np.pi * df[col] / period)
    df[f"{col}_cos"] = np.cos(2 * np.pi * df[col] / period)

data["hour"] = data["time"].dt.hour
data["month"] = data["time"].dt.month
data["dow"] = data["time"].dt.dayofweek

cyclical(data, "hour", 24)
cyclical(data, "month", 12)
cyclical(data, "dow", 7)
data["wind_dir_sin"] = np.sin(np.deg2rad(data["wind_direction_10m"]))
data["wind_dir_cos"] = np.cos(np.deg2rad(data["wind_direction_10m"]))

feature_cols = [
    "pm2_5", "pm10", "carbon_monoxide", "nitrogen_dioxide", "sulphur_dioxide", "ozone",
    "temperature_2m", "relative_humidity_2m", "wind_speed_10m", "surface_pressure",
    "hour_sin", "hour_cos", "month_sin", "month_cos", "dow_sin", "dow_cos",
    "wind_dir_sin", "wind_dir_cos",
]
target_col = "pm2_5"

d:\projects\pm25-forecasting\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
WINDOW = 12
HORIZON = 1

def create_windows(features, target, window=WINDOW, horizon=HORIZON):
    X, y = [], []
    for i in range(len(features) - window - horizon + 1):
        X.append(features[i:i + window])
        y.append(target[i + window + horizon - 1])
    return np.array(X), np.array(y)

X_raw = data[feature_cols].values
y_raw = data[[target_col]].values
X, y = create_windows(X_raw, y_raw)

# Chronological split: 70% train, 10% valid, 20% test
n = len(X)
train_end = int(0.7 * n)
val_end = int(0.8 * n)

X_train, X_val, X_test = X[:train_end], X[train_end:val_end], X[val_end:]
y_train, y_val, y_test = y[:train_end], y[train_end:val_end], y[val_end:]

feature_scaler = StandardScaler().fit(X_train.reshape(-1, X_train.shape[-1]))
X_train_s = feature_scaler.transform(X_train.reshape(-1, X_train.shape[-1])).reshape(X_train.shape)
X_val_s = feature_scaler.transform(X_val.reshape(-1, X_val.shape[-1])).reshape(X_val.shape)
X_test_s = feature_scaler.transform(X_test.reshape(-1, X_test.shape[-1])).reshape(X_test.shape)

target_scaler = StandardScaler().fit(y_train)
y_train_s = target_scaler.transform(y_train)
y_val_s = target_scaler.transform(y_val)
y_test_s = target_scaler.transform(y_test)

def to_dataset(X, y):
    return TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.float32),
    )

train_dataset = to_dataset(X_train_s, y_train_s)
valid_dataset = to_dataset(X_val_s, y_val_s)
test_dataset = to_dataset(X_test_s, y_test_s)

In [3]:
sample0, target0 = train_dataset[0]
sample0.shape, target0.shape

(torch.Size([12, 18]), torch.Size([1]))

In [4]:
batch_size = 64

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

device = "cuda" if torch.cuda.is_available() else "cpu"
n_inputs = train_dataset[0][0].shape[-1]

class LSTMForecaster(nn.Module):
    def __init__(self, n_hidden, n_layers=2, n_inputs=n_inputs, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(n_inputs, n_hidden, n_layers, batch_first=True, dropout=dropout)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(n_hidden, 1)

    def forward(self, X):
        out, _ = self.lstm(X)
        out = self.dropout(out[:, -1, :])
        return self.fc(out)

def evaluate(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred.squeeze(-1), y_batch.squeeze(-1))
    return metric.compute()

In [5]:
def objective(trial):
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-1, log=True)
    n_hidden = trial.suggest_int("n_hidden", 32, 200)
    dropout = trial.suggest_float("dropout", 0.1, 0.5)

    model = LSTMForecaster(n_hidden=n_hidden, dropout=dropout).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    criterion = nn.MSELoss()
    metric = torchmetrics.MeanSquaredError(squared=False).to(device)
    n_epochs = 20

    best_val_rmse = float("inf")
    bad_epochs = 0
    patience = 5

    for epoch in range(n_epochs):
        model.train()
        metric.reset()
        total_loss = 0.

        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred.squeeze(-1), y_batch.squeeze(-1))

        mean_loss = total_loss / len(train_loader)
        train_metric = metric.compute().item()
        valid_metric = evaluate(model, valid_loader, metric).item()

        print(f"Epoch {epoch + 1}/{n_epochs}, "
              f"train loss: {mean_loss:.4f}, "
              f"train RMSE: {train_metric:.4f}, "
              f"valid RMSE: {valid_metric:.4f}")

        # Early stopping: has this trial's own validation RMSE stopped improving?
        if valid_metric < best_val_rmse:
            best_val_rmse = valid_metric
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print(f"Early stopping this trial at epoch {epoch + 1}.")
                break

        # Pruning: is this trial worse than other trials at this point?
        trial.report(valid_metric, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return best_val_rmse

sampler = optuna.samplers.TPESampler(seed=42)
pruner = optuna.pruners.MedianPruner(n_startup_trials=1, n_warmup_steps=5, interval_steps=1)
study = optuna.create_study(direction="minimize", sampler=sampler, pruner=pruner)
study.optimize(objective, n_trials=20)

[I 2026-07-19 14:56:47,479] A new study created in memory with name: no-name-a792fd83-ea64-4e21-b0ca-fc8a14817702


Epoch 1/20, train loss: 0.1477, train RMSE: 0.3845, valid RMSE: 0.2200
Epoch 2/20, train loss: 0.0705, train RMSE: 0.2656, valid RMSE: 0.1970
Epoch 3/20, train loss: 0.0626, train RMSE: 0.2502, valid RMSE: 0.1970
Epoch 4/20, train loss: 0.0582, train RMSE: 0.2412, valid RMSE: 0.1932
Epoch 5/20, train loss: 0.0541, train RMSE: 0.2324, valid RMSE: 0.1931
Epoch 6/20, train loss: 0.0544, train RMSE: 0.2333, valid RMSE: 0.1868
Epoch 7/20, train loss: 0.0523, train RMSE: 0.2286, valid RMSE: 0.1919
Epoch 8/20, train loss: 0.0488, train RMSE: 0.2210, valid RMSE: 0.2045
Epoch 9/20, train loss: 0.0477, train RMSE: 0.2182, valid RMSE: 0.1877
Epoch 10/20, train loss: 0.0499, train RMSE: 0.2235, valid RMSE: 0.1810
Epoch 11/20, train loss: 0.0482, train RMSE: 0.2196, valid RMSE: 0.1882
Epoch 12/20, train loss: 0.0479, train RMSE: 0.2188, valid RMSE: 0.1954
Epoch 13/20, train loss: 0.0477, train RMSE: 0.2184, valid RMSE: 0.1908
Epoch 14/20, train loss: 0.0461, train RMSE: 0.2147, valid RMSE: 0.1921


[I 2026-07-19 14:59:35,178] Trial 0 finished with value: 0.1809851974248886 and parameters: {'learning_rate': 0.0013292918943162175, 'n_hidden': 192, 'dropout': 0.39279757672456206}. Best is trial 0 with value: 0.1809851974248886.


Epoch 15/20, train loss: 0.0446, train RMSE: 0.2112, valid RMSE: 0.1919
Early stopping this trial at epoch 15.
Epoch 1/20, train loss: 0.1114, train RMSE: 0.3339, valid RMSE: 0.2133
Epoch 2/20, train loss: 0.0617, train RMSE: 0.2484, valid RMSE: 0.1954
Epoch 3/20, train loss: 0.0608, train RMSE: 0.2466, valid RMSE: 0.2004
Epoch 4/20, train loss: 0.0529, train RMSE: 0.2298, valid RMSE: 0.2219
Epoch 5/20, train loss: 0.0569, train RMSE: 0.2379, valid RMSE: 0.2573


[I 2026-07-19 14:59:56,190] Trial 1 pruned. 


...

Epoch 10/20, train loss: 0.0429, train RMSE: 0.2071, valid RMSE: 0.1860
Early stopping this trial at epoch 10.
Epoch 1/20, train loss: 0.1430, train RMSE: 0.3783, valid RMSE: 0.2161
Epoch 2/20, train loss: 0.0642, train RMSE: 0.2533, valid RMSE: 0.2738
Epoch 3/20, train loss: 0.0546, train RMSE: 0.2336, valid RMSE: 0.1820
Epoch 4/20, train loss: 0.0574, train RMSE: 0.2395, valid RMSE: 0.1992
Epoch 5/20, train loss: 0.0516, train RMSE: 0.2273, valid RMSE: 0.1867
Epoch 6/20, train loss: 0.0609, train RMSE: 0.2469, valid RMSE: 0.2066
Epoch 7/20, train loss: 0.0555, train RMSE: 0.2357, valid RMSE: 0.2127


[I 2026-07-19 15:17:57,408] Trial 18 finished with value: 0.18200139701366425 and parameters: {'learning_rate': 0.012690764532532676, 'n_hidden': 171, 'dropout': 0.10139033906344114}. Best is trial 11 with value: 0.17507129907608032.


Epoch 8/20, train loss: 0.0555, train RMSE: 0.2356, valid RMSE: 0.1918
Early stopping this trial at epoch 8.
Epoch 1/20, train loss: 0.1912, train RMSE: 0.4374, valid RMSE: 0.2584
Epoch 2/20, train loss: 0.0768, train RMSE: 0.2772, valid RMSE: 0.2147
Epoch 3/20, train loss: 0.0633, train RMSE: 0.2517, valid RMSE: 0.2092
Epoch 4/20, train loss: 0.0587, train RMSE: 0.2424, valid RMSE: 0.1919
Epoch 5/20, train loss: 0.0541, train RMSE: 0.2327, valid RMSE: 0.1960
Epoch 6/20, train loss: 0.0504, train RMSE: 0.2245, valid RMSE: 0.1902
Epoch 7/20, train loss: 0.0520, train RMSE: 0.2280, valid RMSE: 0.2038
Epoch 8/20, train loss: 0.0476, train RMSE: 0.2182, valid RMSE: 0.1972
Epoch 9/20, train loss: 0.0456, train RMSE: 0.2136, valid RMSE: 0.1804
Epoch 10/20, train loss: 0.0446, train RMSE: 0.2113, valid RMSE: 0.1818
Epoch 11/20, train loss: 0.0456, train RMSE: 0.2136, valid RMSE: 0.1803
Epoch 12/20, train loss: 0.0443, train RMSE: 0.2106, valid RMSE: 0.1898
Epoch 13/20, train loss: 0.0429, tra

[I 2026-07-19 15:19:55,534] Trial 19 finished with value: 0.17798727750778198 and parameters: {'learning_rate': 0.000783811055603235, 'n_hidden': 142, 'dropout': 0.2826173426603851}. Best is trial 11 with value: 0.17507129907608032.


Epoch 20/20, train loss: 0.0385, train RMSE: 0.1963, valid RMSE: 0.1780


In [7]:
print("Best params:", study.best_params)
print("Best valid RMSE:", study.best_value)

best_params = study.best_params

Best params: {'learning_rate': 0.0025583692821127078, 'n_hidden': 145, 'dropout': 0.11549007998447333}
Best valid RMSE: 0.17507129907608032


In [10]:
import time
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

final_model = LSTMForecaster(
    n_hidden=best_params["n_hidden"],
    dropout=best_params["dropout"],
).to(device)

optimizer = torch.optim.Adam(
    final_model.parameters(),
    lr=best_params["learning_rate"],
    weight_decay=1e-4,
)
criterion = nn.MSELoss()
n_epochs = 40
patience = 5

# -----------------------------
# Train (with early stopping on valid_loader)
# -----------------------------
train_start = time.time()

best_val_loss = float("inf")
bad_epochs = 0
best_state = None

for epoch in range(n_epochs):
    final_model.train()
    total_loss = 0.

    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        y_pred = final_model(X_batch)
        loss = criterion(y_pred, y_batch)
        total_loss += loss.item()
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

    mean_loss = total_loss / len(train_loader)

    final_model.eval()
    val_loss = 0.
    with torch.no_grad():
        for X_batch, y_batch in valid_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = final_model(X_batch)
            val_loss += criterion(y_pred, y_batch).item()
    val_loss /= len(valid_loader)

    print(f"Epoch {epoch + 1}/{n_epochs}, train loss: {mean_loss:.4f}, valid loss: {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.clone() for k, v in final_model.state_dict().items()}
        bad_epochs = 0
    else:
        bad_epochs += 1
        if bad_epochs >= patience:
            print(f"Early stopping at epoch {epoch + 1}.")
            break

final_model.load_state_dict(best_state)
training_time = time.time() - train_start

# -----------------------------
# Predict on test set
# -----------------------------
final_model.eval()
all_preds, all_true = [], []

start = time.time()
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        y_pred = final_model(X_batch)
        all_preds.append(y_pred.cpu().numpy())
        all_true.append(y_batch.numpy())
inference_time = (time.time() - start) / len(test_dataset)

preds_scaled = np.concatenate(all_preds)
true_scaled = np.concatenate(all_true)

# -----------------------------
# Inverse transform to original PM2.5 scale
# -----------------------------
preds = target_scaler.inverse_transform(preds_scaled)
true = target_scaler.inverse_transform(true_scaled)

# -----------------------------
# Metrics
# -----------------------------
rmse = root_mean_squared_error(true, preds)
mae = mean_absolute_error(true, preds)
r2 = r2_score(true, preds)

print(f"\nRMSE          : {rmse:.4f}")
print(f"MAE           : {mae:.4f}")
print(f"R²            : {r2:.4f}")
print(f"Training time : {training_time:.2f}s")
print(f"Inference time: {inference_time*1_000_000:.4f}µs")

Epoch 1/40, train loss: 0.1149, valid loss: 0.0453
Epoch 2/40, train loss: 0.0519, valid loss: 0.0345
Epoch 3/40, train loss: 0.0519, valid loss: 0.0329
Epoch 4/40, train loss: 0.0473, valid loss: 0.0348
Epoch 5/40, train loss: 0.0442, valid loss: 0.0436
Epoch 6/40, train loss: 0.0437, valid loss: 0.0318
Epoch 7/40, train loss: 0.0426, valid loss: 0.0310
Epoch 8/40, train loss: 0.0431, valid loss: 0.0331
Epoch 9/40, train loss: 0.0420, valid loss: 0.0308
Epoch 10/40, train loss: 0.0402, valid loss: 0.0328
Epoch 11/40, train loss: 0.0396, valid loss: 0.0330
Epoch 12/40, train loss: 0.0413, valid loss: 0.0400
Epoch 13/40, train loss: 0.0369, valid loss: 0.0310
Epoch 14/40, train loss: 0.0376, valid loss: 0.0333
Early stopping at epoch 14.

RMSE          : 10.0260
MAE           : 5.6317
R²            : 0.8875
Training time : 80.70s
Inference time: 105.4723µs
